# ML-08 — First Honest Model vs. Rule Baseline

This notebook preserves our exact Week-5 standardized **Logistic Regression** model trained on `w05_ml_practice_dataset.csv` ($N = 100$) using an 80/20 stratified train/test split (`random_state=42`) and compares it side-by-side against the Week-4 rule baseline (`score >= 3`).

## 1. Load data and create the 80/20 stratified train/test split

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

def find_repo_root() -> Path:
    cur = Path.cwd().resolve()
    for p in [cur, *cur.parents]:
        if (p / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            return p
    return cur

REPO_ROOT = find_repo_root()
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

df = pd.read_csv(REPO_ROOT / "work" / "data" / "w05_ml_practice_dataset.csv")
feature_cols = ["impressions", "clicks", "staleness_days", "position"]
X = df[feature_cols]
y = df["target"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Dataset shape:", df.shape)
print("Train shape:", X_train.shape, "| Positive rate:", round(float(y_train.mean()), 4))
print("Test shape :", X_test.shape,  "| Positive rate:", round(float(y_test.mean()), 4))


Dataset shape: (100, 6)
Train shape: (80, 4) | Positive rate: 0.6125
Test shape : (20, 4) | Positive rate: 0.6


## 2. Evaluate the Week-4 Rule Baseline (`score >= 3`) on the Test Split

In [2]:
baseline_score = (
    (X_test["impressions"] >= 1000).astype(int) * 2
    + (X_test["staleness_days"] >= 14).astype(int) * 2
    + (X_test["position"] >= 8).astype(int) * 1
)
baseline_pred = (baseline_score >= 3).astype(int)

print("Week-4 Baseline Test Metrics (n=20):")
print("  Accuracy :", round(float(accuracy_score(y_test, baseline_pred)), 4))
print("  F1 Score :", round(float(f1_score(y_test, baseline_pred)), 4))
print("  ROC AUC  :", round(float(roc_auc_score(y_test, baseline_score)), 4))


Week-4 Baseline Test Metrics (n=20):
  Accuracy : 0.7
  F1 Score : 0.7857
  ROC AUC  : 0.8542


## 3. Train Standardized Logistic Regression & Compare Side-by-Side

In [3]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

comparison = pd.DataFrame([
    {
        "Method": "Week-4 Baseline (score >= 3)",
        "Accuracy": round(float(accuracy_score(y_test, baseline_pred)), 4),
        "Precision": round(float(precision_score(y_test, baseline_pred)), 4),
        "Recall": round(float(recall_score(y_test, baseline_pred)), 4),
        "F1_Score": round(float(f1_score(y_test, baseline_pred)), 4),
        "ROC_AUC": round(float(roc_auc_score(y_test, baseline_score)), 4),
    },
    {
        "Method": "Week-5 Logistic Regression (4 features)",
        "Accuracy": round(float(accuracy_score(y_test, y_pred)), 4),
        "Precision": round(float(precision_score(y_test, y_pred)), 4),
        "Recall": round(float(recall_score(y_test, y_pred)), 4),
        "F1_Score": round(float(f1_score(y_test, y_pred)), 4),
        "ROC_AUC": round(float(roc_auc_score(y_test, y_prob)), 4),
    },
])
print("Side-by-Side Comparison on Held-Out Test Split (n=20, Base Rate = 0.6000):")
print(comparison.to_string(index=False))


Side-by-Side Comparison on Held-Out Test Split (n=20, Base Rate = 0.6000):
                                 Method  Accuracy  Precision  Recall  F1_Score  ROC_AUC
           Week-4 Baseline (score >= 3)       0.7     0.6875  0.9167    0.7857   0.8542
Week-5 Logistic Regression (4 features)       0.8     0.7500  1.0000    0.8571   0.9375


## 4. Standardized Coefficients & Model Interpretation

In [4]:
coefs = model.named_steps["model"].coef_[0]
intercept = float(model.named_steps["model"].intercept_[0])
coef_df = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": np.round(coefs, 6),
    "abs_coefficient": np.round(np.abs(coefs), 6),
}).sort_values("abs_coefficient", ascending=False)

print("Intercept:", round(intercept, 6))
print(coef_df.to_string(index=False))


Intercept: 0.524358
       feature  coefficient  abs_coefficient
staleness_days     0.640502         0.640502
   impressions     0.432939         0.432939
        clicks     0.113056         0.113056
      position     0.104730         0.104730


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`